### Model and experiment parameters

This block defines all parameters used in the numerical experiments for the Kou jump-diffusion model.

First, the model parameters are specified: the jump intensity $\lambda$, the probability $p$ of an upward jump, and the exponential rates $\lambda_+$ and $\lambda_-$ that determine the distributions of positive and negative jump sizes.  
Next, the market and option parameters are fixed, namely the diffusion volatility $\sigma$, the risk-free interest rate $r$, the maturity $T$, the strike price $K$, and the damping parameter $z$ used later in the Fourier pricing formula.

The random seed is chosen to ensure reproducibility of the results.  
Then the simulation parameters for the direct Monte Carlo and hybrid Monte Carlo methods are introduced, including the total number of simulated paths and the confidence level for the statistical error estimate.  
Finally, the training parameters of the neural approximation and the reference and coarse grid parameters are specified for the FFT-based computations and comparison experiments.

In [ ]:
# --- Model (Kou) primary params ---
intensity = 2.0
p = 0.1
lp = 10.0      # Lambda_+ > 0
lm = 15.0      # Lambda_- > 0

# --- Market/option params ---
sigma = 0.3
r = 0.05
T = 1/2
K = 100
z = -2

# --- Seeds ---
seed = 42

# --- Direct MC params ---
lambda_p = lp
lambda_m = lm
M = 400_000
confidence = 0.95

# --- Hybrid MC params ---
L = M

# --- Train parameters ---
N = 16
epochs = 2000

# --- Grid parameters ---
m_ref = 18
d_ref = 0.0001

d5e3 = 0.005
d1e3 = 0.001
d5e4 = 0.0005

m12 = 12
m13 = 13
m14 = 14
m15 = 15
m16 = 16
m17 = 17

### Report logging

The following code block creates a simple logging mechanism for the experiment.  
A text report file is generated automatically, with the filename containing the main experiment parameters (`N`, `epochs`, `L`).  

To ensure that all console outputs are saved, a custom `Tee` class is implemented.  
This class redirects the standard output stream (`stdout`) simultaneously to the console and to the report file. As a result, every `print()` statement executed later in the notebook will appear both in the notebook output and in the saved `.txt` report.

This approach allows us to automatically collect experiment results, diagnostic messages, and intermediate outputs in a single file that can be downloaded from Google Colab.

In [ ]:
import sys
from datetime import datetime
from google.colab import files

REPORT_PATH = f"kou_N={N}_EP={epochs}_L={L}_p={p}.txt"

class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()
    def flush(self):
        for s in self.streams:
            s.flush()

_report_f = open(REPORT_PATH, "w", encoding="utf-8")

_old_stdout = sys.stdout
sys.stdout = Tee(sys.stdout, _report_f)

print("=== Kou report ===")
print("Generated:", datetime.now().isoformat(timespec="seconds"))
print()

=== Kou report ===
Generated: 2026-03-12T17:22:56



# 0. Theory and definitions

Kou jump–diffusion parameters (article parameterization):

$$
\lambda > 0,
\qquad
p \in (0,1),
\qquad
\Lambda_+ > 0,
\qquad
\Lambda_- > 0.
$$

The log-return process is defined as

$$
X_t = \mu t + \sigma W_t + \sum_{k=1}^{N_t} Y_k,
$$

where $W_t$ is a standard Brownian motion, $N_t$ is a Poisson process with intensity $\lambda$, and the jump sizes $Y_k$ follow the asymmetric double-exponential distribution

$$
Y \mid (Y>0) \sim \mathrm{Exp}(\Lambda_+),
\qquad
Y \mid (Y<0) \sim -\mathrm{Exp}(\Lambda_-).
$$

The probabilities of upward and downward jumps are

$$
\mathbb{P}(Y>0)=p,
\qquad
\mathbb{P}(Y<0)=1-p.
$$

For convenience we introduce

$$
c_+ = \lambda p,
\qquad
c_- = \lambda(1-p).
$$

The characteristic exponent of the Kou process is

$$
\psi(\xi)
=
\frac{\sigma^2}{2}\xi^2
-i\mu\xi
+
c_-\,\frac{i\xi}{\Lambda_-+i\xi}
+
c_+\,\frac{i\xi}{-\Lambda_+ + i\xi}.
$$

The characteristic function of the log-return is

$$
\varphi_{X_t}(\xi)=\mathbb{E}[e^{i\xi X_t}]
= e^{-t\psi(\xi)}.
$$

Under the risk-neutral measure the drift parameter $\mu$ is chosen so that

$$
r + \psi(-i) = 0.
$$

The logistic sigmoid is
$$
\sigma(x)=\frac{1}{1+e^{-x}}.
$$
It maps $\mathbb{R}$ to $(0,1)$, is smooth and strictly increasing, and is therefore convenient for building bounded monotone approximations (such as a CDF) using sums of sigmoid “neurons”.

The inverse of the sigmoid is the logit transform:
$$
\operatorname{logit}(u)=\ln\!\left(\frac{u}{1-u}\right), \qquad u\in(0,1).
$$
These are exact inverses:
$$
\sigma(\operatorname{logit}(u))=u,
\qquad
\operatorname{logit}(\sigma(x))=x.
$$

In the later hybrid Monte Carlo step, drawing $u\sim \mathrm{Uniform}(0,1)$ and setting
$$
y=\operatorname{logit}(u)
$$
produces a random variable $y$ with the standard logistic distribution (whose CDF is $\sigma$), which naturally matches the sigmoid-based neural network representation used to approximate the CDF.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import ifft, fftfreq
import cmath
import math
import tensorflow as tf
from google.colab import files
import time
from scipy.stats import t

def sigmoid(x):
  return 1 / (1 + math.exp(-x))

def inverse_logistic(u):
  return np.log(u/(1-u))

### Kou model characteristic exponent and function

In this block we implement the analytical components of the Kou jump–diffusion model.

First, the function `compute_mu` calculates the drift parameter $\mu$ under the risk-neutral measure.  
In the Kou model the jump size $Y$ follows a double-exponential distribution, where upward jumps occur with probability $p$ and rate $\Lambda_{+}$, while downward jumps occur with probability $1-p$ and rate $\Lambda_{-}$.

The expectation of the exponential jump factor is

$$
\mathbb{E}[e^{Y}]
=
p \frac{\Lambda_{+}}{\Lambda_{+}-1}
+
(1-p) \frac{\Lambda_{-}}{\Lambda_{-}+1}.
$$

The risk-neutral drift is then obtained from the martingale condition for the discounted asset price,

$$
\mu
=
r
-
\frac{1}{2}\sigma^2
-
\lambda \left(\mathbb{E}[e^{Y}] - 1\right),
$$

where $\lambda$ is the jump intensity.

Next, the function `kou_chexp` evaluates the **characteristic exponent** of the Kou process.  
For the jump–diffusion model with double-exponential jumps it has the form

$$
\psi(\xi)
=
\frac{1}{2}\sigma^2 \xi^2
-
i \mu \xi
+
\frac{c_+\, i\xi}{i\xi - \Lambda_{+}}
+
\frac{c_-\, i\xi}{i\xi + \Lambda_{-}},
$$

where

$$
c_+ = \lambda p,
\qquad
c_- = \lambda (1-p).
$$

The first two terms correspond to the diffusion part of the process, while the remaining rational terms describe the contribution of upward and downward jumps.

Finally, the function `kou_chf` computes the **characteristic function** of the log-return $X_t$,

$$
\varphi_{X_t}(\xi)
=
\mathbb{E}[e^{i\xi X_t}]
=
\exp\!\left(-t\,\psi(\xi)\right).
$$

This characteristic function is later used in the Fourier inversion procedures for computing the reference distribution and for FFT-based option pricing.

In [ ]:
def compute_mu(r, sigma, intensity, lp, lm, p):
    EY_exp = (
        p * lp / (lp - 1) +
        (1 - p) * lm / (lm + 1)
    )
    mu = r - 0.5 * sigma**2 - intensity * (EY_exp - 1)
    return mu


def kou_chexp(xi, sigma, cp, cm, lm, lp, mu):
    sig2 = 0.5 * sigma ** 2
    temp1 = sig2 * xi ** 2 - 1j * mu * xi
    temp2 = (cp * 1j * xi) / (1j * xi - lp)
    temp3 = (cm * 1j * xi) / (1j * xi + lm)
    return temp1 + temp2 + temp3

def kou_chf(xi, t, sigma, cp, cm, lm, lp, mu):
    temp1 = kou_chexp(xi, sigma, cp, cm, lm, lp, mu)
    return cmath.exp(-t * temp1)

### FFT-based construction of the reference CDF

In this block we construct a numerical approximation of the cumulative distribution function of the Kou log-return by Fourier inversion on a uniform grid.

First, the jump parameters are rewritten as

$$
c_+ = \lambda p,
\qquad
c_- = \lambda(1-p),
$$

and the risk-neutral drift $\mu$ is computed from the martingale condition.  
Then the number of grid points is chosen as

$$
M = 2^m,
$$

which is convenient for the use of the Fast Fourier Transform.

Next, a uniform frequency grid is introduced.  
Its initial value is taken as

$$
\xi_{\max} = \frac{\pi}{d},
$$

and the step size is

$$
\Delta \xi = -\frac{2\xi_{\max}}{M}.
$$

At each frequency point, the characteristic function of the Kou model,

$$
\varphi_{X_T}(\xi)
=
\exp\!\left(-T\,\psi(\xi)\right),
$$

is evaluated, where $\psi(\xi)$ is the characteristic exponent of the jump-diffusion process.  
An alternating factor $(-1)^l$ is included in the sampled values, which corresponds to the standard FFT shift and ensures correct centering of the spatial grid after inversion.

After that, the inverse Fast Fourier Transform is applied to the sampled characteristic function values.  
Taking the real part yields a numerical approximation of the density on the spatial grid.  
Because of the FFT ordering, the values at odd indices are multiplied by $-1$ in order to restore the correct alignment of the density.

Once the density approximation is obtained, the cumulative distribution function is formed by discrete summation along the grid,

$$
F(x_k)
\approx
\sum_{j=0}^{k} f(x_j),
\qquad k = 0,1,\dots,M-1.
$$

Finally, the corresponding spatial grid is defined as

$$
x_k \in \left[-\frac{Md}{2}, \frac{Md}{2}\right],
\qquad k = 0,1,\dots,M-1,
$$

with uniform spacing $d$.  
The function returns both the approximated CDF values and the associated grid points.  
This reference distribution is later used for interpolation, neural-network training, and benchmarking of the pricing methods.

In [ ]:
def compute_cdf(m: int, d: float, sigma: float, p: float, r: float,
                intensity: float, lp: float, lm: float, T: float):

    cp, cm = intensity * p, intensity * (1 - p)
    mu = compute_mu(r, sigma, intensity, lp, lm, p)
    M = 2 ** m

    xi = np.pi / d
    dxi = -2 * xi / M
    x = M * d / 2.0

    chf = np.zeros(M, dtype=complex)
    sign = 1
    for l in range(M):
        phi = kou_chf(xi, T, sigma, cp, cm, lm, lp, mu)
        chf[l] = sign * phi
        xi += dxi
        sign *= -1

    dens = np.real(np.fft.ifft(chf))
    odd_indices = np.arange(1, len(dens), 2)
    dens[odd_indices] *= -1
    x = np.linspace(-M * d / 2, M * d / 2, M)

    cdf = np.zeros_like(dens)
    x = -(M // 2) * d
    cdf[0] = sign * dens[0]
    for k in range(1, M):
        x += d
        cdf[k] = cdf[k-1] + sign * dens[k]

    x = np.linspace(-M * d / 2, M * d / 2, M)

    return cdf, x


### Neural approximation of the reference CDF

In this block we construct and train a neural-network approximation of the reference cumulative distribution function obtained from the Fourier inversion procedure.

The input training set consists of pairs $(x_i, F(x_i))$, where $x_i$ are grid points and $F(x_i)$ are the corresponding values of the reference CDF.  
The goal is to approximate this function by a one-hidden-layer neural network of the form

$$
\widehat{F}(x)
=
\sum_{j=1}^{N} p_j \, \sigma(\alpha_j x + \beta_j),
$$

where $\sigma(u) = \dfrac{1}{1+e^{-u}}$ is the sigmoid activation function, $\alpha_j$ are the first-layer weights, $\beta_j$ are the biases, and $p_j$ are the output-layer weights.

The architecture is chosen so that the approximation has the structure of a weighted sum of sigmoid functions.  
Such a representation is convenient because it can be interpreted as a smooth approximation of a cumulative distribution function.

An additional lower bound is imposed on the first-layer weights,

$$
\alpha_j \ge 1.
$$

This restriction is introduced for numerical stability in the hybrid Monte Carlo sampling procedure used later.  
In the sampling algorithm, the inverse transformation of the sigmoid argument is applied.  
Specifically, random variables are generated as

$$
y = \log \frac{u}{1-u}, \qquad u \sim U(0,1),
$$

and then transformed into the log-return variable through

$$
z = \frac{y - \beta_j}{\alpha_j}.
$$

Thus, the parameter $\alpha_j$ appears in the denominator of the transformation.  
If $\alpha_j$ becomes very small, the resulting values of $z$ can become extremely large in magnitude.  
Since the terminal asset price is computed as

$$
S_T = e^{z},
$$

even moderately large values of $z$ lead to extremely large asset prices and therefore unrealistic option payoffs.

To avoid such numerical explosions in the hybrid Monte Carlo estimator, the weights of the first layer are constrained so that

$$
\alpha_j \ge 1,
$$

which keeps the transformation stable and prevents the generation of excessively large simulated prices.
The weights of the second layer represent probabilities associated with each sigmoid component.  
Therefore they must satisfy the conditions

$$
p_j \ge 0,
\qquad
\sum_{j=1}^{N} p_j = 1.
$$

This constraint guarantees that the neural approximation has the structure of a convex combination of sigmoid functions,

$$
\widehat{F}(x)
=
\sum_{j=1}^{N} p_j \, \sigma(\alpha_j x + \beta_j).
$$

Such a representation naturally resembles a mixture model, where each sigmoid component contributes to the total approximation with probability weight $p_j$.  
Because the weights are nonnegative and normalized, the approximation remains bounded and preserves the qualitative behavior expected from a cumulative distribution function.

As a result, the final approximation remains a convex combination of monotone sigmoid functions, which is consistent with the structure of a valid CDF.

The model is trained by minimizing the mean squared error

$$
\mathrm{MSE}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\widehat{F}(x_i) - F(x_i)
\right)^2.
$$

Before training, the random seed is fixed in both TensorFlow and NumPy in order to make the experiment reproducible.  
During optimization, the training loss is periodically printed, which allows us to monitor convergence.  
The training time is also measured, since runtime is one of the quantities compared in the numerical experiments.

After training, the reference CDF is recomputed on a finer grid and the neural network is evaluated on this grid.  
This produces the approximation curve

$$
x \mapsto \widehat{F}(x),
$$

which is then plotted together with the reference CDF for visual comparison.  
The figure also reports the grid parameters $m$ and $d$, the number of hidden units $N$, and the training time.

Finally, the trained network parameters are extracted explicitly:
the first-layer weights $\alpha_j$, the biases $\beta_j$, and the second-layer weights $p_j$.  
These quantities define the neural approximation completely and are later used in the hybrid Monte Carlo sampling procedure.  
Thus, this block provides the transition from the FFT-based reference distribution to its compact neural representation.

In [ ]:
def compute_approximation(x_train, y_train, epochs, batch_size, N, m, d,
                          m_ref, d_ref,
                          sigma, p, r, intensity, lp, lm, T,
                          seed):
    print(len(x_train))

    tag = CURRENT_MODEL_TAG if CURRENT_MODEL_TAG else f"m={m}, d={d}, batch_size={batch_size}"

    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)

    class GreaterEqualOne(tf.keras.constraints.Constraint):
        def __call__(self, w):
            return tf.maximum(w, 1.0)

    class EpochLogger(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            if logs is None:
                return
            if (epoch + 1) % 500 == 0 or epoch == 0:
                print(f"Epoch {epoch + 1}: loss = {logs['loss']}")

    class SumToOne(tf.keras.constraints.Constraint):
        def __init__(self, axis=0):
            self.axis = axis

        def __call__(self, w):
            w = tf.nn.relu(w)
            return w / (tf.reduce_sum(w, axis=self.axis, keepdims=True) + tf.keras.backend.epsilon())

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(
            N,
            input_shape=(1,),
            activation='sigmoid',
            kernel_constraint=GreaterEqualOne(),
            use_bias=True
        ),
        tf.keras.layers.Dense(
            1,
            activation='linear',
            kernel_constraint=SumToOne(),
            use_bias=False
        )
    ])

    model.compile(optimizer='adam', loss='mse', jit_compile=True)

    start_time = time.perf_counter()
    history = model.fit(
        x_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        verbose=0,
        callbacks=[EpochLogger()]
    )
    end_time = time.perf_counter()

    losses = history.history.get("loss", [])
    if len(losses) > 0:
        print(f"batch_size = {batch_size}, epochs = {epochs}, N = {N}")
        print(f"loss min   = {min(losses):.12e}")
    print(f"train_time = {end_time - start_time:.6f} sec")
    print()

    cdf_ref, x_ref = compute_cdf(m_ref, d_ref, sigma, p, r, intensity, lp, lm, T)

    x_test = x_ref.reshape(-1, 1)
    y_pred = model.predict(x_test, verbose=0)

    plt.figure(figsize=(8, 6))
    plt.plot(x_ref, cdf_ref, label='Referenced cdf', color='blue')
    plt.plot(x_test, y_pred, label='Approximation curve', color='red')
    plt.xlim(-2, 2)

    plt.title('Cumulative Distribution Function')
    plt.xlabel('x')
    plt.ylabel('CDF')
    plt.legend()
    plt.grid(True)

    textstr = (
        rf"$m = {m}$" "\n"
        rf"$d = {d}$" "\n"
        rf"$N = {N}$" "\n"
        rf"$T_{{train}} = {end_time - start_time:.4f}\,\mathrm{{s}}$"
    )

    plt.text(
        0.02, 0.98, textstr,
        transform=plt.gca().transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

    filename = f"approximated_cdf_{m}_{d}_{N}_{epochs}.pdf"
    plt.savefig(filename, format="pdf", bbox_inches="tight", dpi=500)
    files.download(filename)
    plt.show()

    weights, biases = model.layers[0].get_weights()
    alphas = weights.T.flatten().tolist()
    betas = biases.tolist()

    print(f"First layer weights (alphas):")
    for a in alphas:
        print(a)
    print()

    print(f"First layer biases (betas):")
    for b in betas:
        print(b)
    print()

    weights2 = model.layers[1].get_weights()[0]
    probabilities = weights2.T.flatten().tolist()

    print(f"Second layer weights (probabilities):")
    for pp in probabilities:
        print(pp)
    print()

    train_time = end_time - start_time
    return alphas, betas, probabilities, train_time, model

### Error analysis of the neural CDF approximation

In this block we evaluate the approximation error of the trained neural network with respect to the reference cumulative distribution function.

Since the neural network and the reference CDF may originally be defined on different grids, we first restrict attention to their common domain of definition.  
More precisely, if the training grid is denoted by $x_{\mathrm{train}}$ and the reference grid by $x_{\mathrm{ref}}$, then the comparison is performed on the interval

$$
[x_{\min}, x_{\max}],
\qquad
x_{\min} = \max\bigl(\min x_{\mathrm{ref}}, \min x_{\mathrm{train}}\bigr),
\qquad
x_{\max} = \min\bigl(\max x_{\mathrm{ref}}, \max x_{\mathrm{train}}\bigr).
$$

Only the reference grid points belonging to this common interval are retained for the error computation.

Next, the trained neural network is evaluated at these points, producing the approximation values

$$
\widehat{F}(x_i),
$$

while the reference CDF provides the benchmark values

$$
F_{\mathrm{ref}}(x_i).
$$

The pointwise approximation error is then defined as

$$
e(x_i)
=
\widehat{F}(x_i) - F_{\mathrm{ref}}(x_i).
$$

Thus, the function computes the full error curve over the common grid and stores all pointwise differences.

After that, the maximal pointwise deviation is identified.  
In the present implementation, the reported quantity is the largest positive error,

$$
\max_i e(x_i),
$$

together with the grid point at which this maximum is attained.  
This allows us to detect where the neural approximation deviates most strongly from the reference distribution.

For visualization, the error curve $e(x)$ is plotted over the spatial variable $x$.  
The plot provides a direct picture of the approximation quality across the relevant region of the state space and shows whether the network systematically overestimates or underestimates the reference CDF.

The figure also includes the numerical parameters $m$, $d$, and $N$, as well as the reported maximal error value.  
Finally, the error plot is saved as a PDF file, and the function returns both the vector of pointwise errors and the corresponding grid points.  
This block is therefore used to assess the quality of the neural approximation before the extracted parameters are employed in the hybrid Monte Carlo pricing procedure.

In [ ]:
def compute_cdf_error(x_train, model, ref_x, ref_cdf, m, d, N):

    xmin = max(ref_x.min(), x_train.min())
    xmax = min(ref_x.max(), x_train.max())

    mask = (ref_x >= xmin) & (ref_x <= xmax)
    err_x = ref_x[mask].reshape(-1, 1)
    ref_eval = ref_cdf[mask]

    nn_eval = model.predict(err_x, verbose=0).reshape(-1)

    error = np.zeros_like(ref_eval)

    max_err = 0.0
    idx = -1

    for i in range(len(ref_eval)):
        err = nn_eval[i] - ref_eval[i]
        error[i] = err
        if err > max_err:
            max_err = err
            idx = i

    print("max (CDF_NN(x) - CDF_ref(x)) =", max_err)
    print("at x =", float(err_x[idx]))

    plt.figure(figsize=(8, 6))
    plt.plot(err_x, error, label='Error curve', color='red')
    plt.xlim(-2, 2)

    plt.title('CDF Error')
    plt.xlabel('x')
    plt.ylabel('CDF Error')
    plt.legend()
    plt.grid(True)

    textstr = (
        rf"$m = {m}$" "\n"
        rf"$d = {d}$" "\n"
        rf"$N = {N}$" "\n"
        rf"$\max\_error = {max_err:.6g}$"
    )

    plt.text(
        0.02, 0.98, textstr,
        transform=plt.gca().transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )

    filename = f"cdf_error_{m}_{d}_{N}.pdf"
    plt.savefig(
        filename,
        format="pdf",
        bbox_inches="tight",
        dpi=300
    )
    files.download(filename)
    plt.show()

    return error, err_x

### FFT-based option pricing under the Kou model

In this block we implement the FFT-based pricing formula for a European call option under the Kou jump-diffusion model.

The method is based on Fourier inversion of the damped call payoff transform.  
Since the call payoff is not integrable over the whole real line, a damping parameter $z$ is introduced, and the characteristic exponent is evaluated at the complex argument

$$
\xi + i z.
$$

First, the jump coefficients are written as

$$
c_+ = \lambda p,
\qquad
c_- = \lambda(1-p),
$$

and the drift parameter $\mu$ is computed under the risk-neutral measure.  
Using these quantities, the shifted characteristic exponent of the Kou model is

$$
\psi_z(\xi)
=
\frac{\sigma^2}{2}(\xi + iz)^2
-
i\mu(\xi + iz)
+
c_+\,\frac{i(\xi + iz)}{i(\xi + iz)-\Lambda_+}
+
c_-\,\frac{i(\xi + iz)}{i(\xi + iz)+\Lambda_-}.
$$

The corresponding damped characteristic function is then

$$
\varphi_z(\xi)
=
\exp\!\left(-T\,\psi_z(\xi)\right).
$$

Next, the Fourier grid is introduced with

$$
M = 2^m,
$$

which allows efficient use of the inverse Fast Fourier Transform.  
The frequency points are sampled on the interval determined by

$$
\xi_{\min} = -\frac{\pi}{d},
\qquad
\Delta \xi = \frac{2\pi}{Md}.
$$

At each frequency point, the transformed pricing integrand is evaluated as

$$
\frac{\varphi_z(\xi)}
{(\xi + iz)(\xi + i(z+1))}.
$$

The denominator comes from the Fourier transform of the damped call payoff.  
As in the previous FFT constructions, an alternating factor $(-1)^k$ is included to implement the standard FFT shift and align the spatial grid correctly.

After assembling these Fourier samples, the inverse FFT is applied.  
This yields the transformed option values on the corresponding log-moneyness grid.

The target point is determined by the log-moneyness

$$
x_{\mathrm{target}} = \ln\frac{S}{K},
$$

where $S$ is the current asset price and $K$ is the strike.  
The closest grid index $j$ is selected, and if $x_j$ is the corresponding spatial point, then the option price is recovered by the inverse damping transformation:

$$
V(S)
=
(-1)^{j+1}
\, \Re(v_x[j])
\, K
\, e^{-z x_j}
\, e^{-rT}
\, \frac{1}{d}.
$$

Thus, this block computes the European call price directly from the characteristic function of the Kou jump-diffusion log-return.  
The resulting FFT price is later used both as a benchmark and as one of the methods in the numerical comparison tables.

In [ ]:
def compute_price_fft(m: int, d: float, S: float,
                      sigma: float, p: float, r: float,
                      intensity: float, lp: float, lm: float,
                      T: float, K: float, z: float) -> float:

    cp, cm = intensity * p, intensity * (1 - p)
    def kou_chexp_fft(xi, z, sigma, cp, cm, lm, lp, mu):
      xi = xi + z * 1j
      sig2 = 0.5 * sigma ** 2
      temp1 = sig2 * xi ** 2 - 1j * mu * xi
      temp2 = (cp * 1j * xi) / (1j * xi - lp)
      temp3 = (cm * 1j * xi) / (1j * xi + lm)
      return temp1 + temp2 + temp3

    def kou_chf_fft(xi, z, t, sigma, cp, cm, lm, lp, mu):
        temp1 = kou_chexp_fft(xi, z, sigma, cp, cm, lm, lp, mu)
        return cmath.exp(-t * temp1)

    mu = compute_mu(r, sigma, intensity, lp, lm, p)

    M = int(2 ** m)

    xi = -np.pi / d
    dxi = -2 * xi / M

    v_F = np.zeros(M, dtype=complex)

    sign_out = 1
    for k in range(M):
        phi = kou_chf_fft(xi, z, T, sigma, cp, cm, lm, lp, mu)
        F = sign_out * phi
        v_F[k] = F / ((xi + 1j * z) * (xi + 1j * (z + 1)))

        xi += dxi
        sign_out = -sign_out

    v_x = np.fft.ifft(v_F)

    x_target = np.log(S / K)
    j = int(M / 2 + x_target / d)
    x = -M * d / 2 + j * d

    price = ((-1) ** (j + 1)) * v_x[j].real * K * np.exp(-x * z) * np.exp(-r * T) / d
    return float(price)

### Direct Monte Carlo pricing under the Kou model

In this block we implement the direct Monte Carlo estimator for the price of a European call option under the Kou jump-diffusion model.

In the Kou model, the log-return at maturity is represented as

$$
X_T
=
\mu T + \sigma W_T + \sum_{k=1}^{N_T} Y_k,
$$

where $W_T \sim \mathcal{N}(0,T)$ is the diffusion part, $N_T \sim \mathrm{Poisson}(\lambda T)$ is the number of jumps up to time $T$, and the jump sizes $Y_k$ follow the asymmetric double-exponential law.  
Upward jumps occur with probability $p$ and are sampled from an exponential distribution with rate $\Lambda_+$, while downward jumps occur with probability $1-p$ and are sampled as minus an exponential random variable with rate $\Lambda_-$.

The drift parameter $\mu$ is computed from the risk-neutral martingale condition.  
Then, for each simulated path, the algorithm generates the Brownian increment $W_T$, the number of jumps $N_T$, and the total jump contribution

$$
J = \sum_{k=1}^{N_T} Y_k.
$$

If jumps occur, each jump direction is first selected by a Bernoulli trial with probability $p$, after which positive and negative jumps are sampled separately and added together.  
The terminal stock price is then constructed as

$$
S_T = S_0 \exp\!\left(\mu T + \sigma W_T + J\right).
$$

For each path, the discounted payoff of the European call option is computed as

$$
\Pi = e^{-rT}\max(S_T - K, 0).
$$

The Monte Carlo price estimator is the sample mean of these discounted payoffs,

$$
\widehat{V}_{MC}
=
\frac{1}{M}\sum_{i=1}^{M} \Pi^{(i)}.
$$

To quantify the simulation uncertainty, the sample standard deviation of the discounted payoffs is computed and combined with the Student $t$-quantile.  
If $\widehat{\sigma}$ is the sample standard deviation, then the confidence error estimate is

$$
\mathrm{error}
=
t_{(1+\mathrm{confidence})/2,\,M-1}
\frac{\widehat{\sigma}}{\sqrt{M}}.
$$

Thus, this block provides a direct simulation-based benchmark for pricing under the Kou model.  
Unlike the FFT-based method, it does not rely on Fourier inversion, and unlike the hybrid Monte Carlo method, it simulates the jump-diffusion process directly from its probabilistic construction.

In [ ]:
def compute_price_MC_direct(S0, K, r, T, sigma,
                            intensity, p, lambda_p, lambda_m,
                            M, confidence, seed):

    rng = np.random.default_rng(seed)

    mu = compute_mu(r, sigma, intensity, lambda_p, lambda_m, p)
    W_T = rng.normal(0.0, np.sqrt(T), M)
    N = rng.poisson(intensity * T, M)
    J = np.zeros(M)

    for i in range(M):
        if N[i] > 0:
            directions = rng.random(N[i]) < p

            pos_jumps = rng.exponential(1 / lambda_p, np.sum(directions))
            neg_jumps = -rng.exponential(1 / lambda_m, np.sum(~directions))

            J[i] = pos_jumps.sum() + neg_jumps.sum()

    S_T = S0 * np.exp(mu * T + sigma * W_T + J)
    payoff = np.exp(-r * T) * np.maximum(S_T - K, 0)
    mean = payoff.mean()
    std = payoff.std(ddof=1)
    t_value = t.ppf(0.5+confidence/2, M - 1)
    error = t_value * std / np.sqrt(M)

    return mean, error

### Hybrid Monte Carlo pricing based on the neural CDF approximation

In this block we implement the hybrid Monte Carlo pricing method based on the neural approximation of the cumulative distribution function.

The trained neural network represents the CDF as a convex combination of sigmoid functions,

$$
\widehat{F}(x)
=
\sum_{j=1}^{N} p_j \, \sigma(\alpha_j x + \beta_j),
$$

where $p_j$ are nonnegative weights such that

$$
p_j \ge 0,
\qquad
\sum_{j=1}^{N} p_j = 1.
$$

Because of this form, the approximation can be interpreted as a mixture representation.  
Each component is selected with probability $p_j$, and then a sample is generated conditionally on that component.

For a fixed component $j$, the sigmoid function is inverted through the logistic transform.  
If

$$
u \sim U(0,1),
$$

then

$$
y = \log \frac{u}{1-u}
$$

has the logistic distribution.  
Using the parameters of the $j$-th sigmoid, this variable is transformed into

$$
z = \frac{y - \beta_j}{\alpha_j}.
$$

This quantity represents a simulated value of the log-return in normalized coordinates.  
The corresponding normalized terminal asset price is

$$
\widetilde{S}_T = e^z.
$$

Since the code works with the normalized strike

$$
\widetilde{K} = \frac{K}{S_0},
$$

the simulated payoff is computed as

$$
\max(\widetilde{S}_T - \widetilde{K}, 0).
$$

The algorithm allocates approximately

$$
n_j \approx p_j L
$$

samples to each component, where $L$ is the total Monte Carlo budget.  
Thus, components with larger probability weights contribute more samples to the estimator.

After generating all samples, the discounted option price is estimated as

$$
\widehat{V}_{\mathrm{HMC}}
=
S_0 e^{-rT}
\frac{1}{L}
\sum_{j=1}^{N}
\sum_{i=1}^{n_j}
\max(\widetilde{S}_T^{(i,j)} - \widetilde{K}, 0).
$$

To assess the statistical uncertainty, the second empirical moment of the discounted payoff is also accumulated.  
This gives the variance estimator

$$
\widehat{\mathrm{Var}}
=
\frac{\widehat{\mathbb{E}[\Pi^2]} - \widehat{V}_{\mathrm{HMC}}^2}{L-1},
$$

where $\Pi$ denotes the discounted payoff.  
The Monte Carlo error is then approximated by the normal-confidence formula

$$
\mathrm{mc\_error}
=
1.96 \sqrt{\max(\widehat{\mathrm{Var}}, 0)}.
$$

Therefore, this method combines two ingredients:  
the FFT-based reference CDF is first compressed into a neural mixture representation, and then this representation is used for fast sampling in the pricing stage.  
In this sense, the procedure is hybrid: it is neither a purely deterministic Fourier method nor a purely direct Monte Carlo simulation from the original Kou process.

In [ ]:
def compute_price_hybrid(S_0, alphas, betas, probabilities,
                         K, T, L, r, seed):

    rng = np.random.default_rng(seed)

    norK = K / S_0
    N = len(probabilities)

    payoff_sum = 0.0
    payoff_sq_sum = 0.0
    total = 0

    for j in range(N):
        n_j = int(probabilities[j] * L)
        if n_j <= 0:
            continue

        u = rng.random(n_j)
        y = np.log(u / (1.0 - u))
        z = (y - betas[j]) / alphas[j]
        S_T = np.exp(z)
        poff = np.maximum(S_T - norK, 0.0)

        payoff_sum += poff.sum()
        payoff_sq_sum += np.square(poff).sum()
        total += n_j

    V = S_0 * np.exp(-r * T) * payoff_sum / L

    second_moment = (S_0 ** 2) * np.exp(-2 * r * T) * payoff_sq_sum / L
    Var = (second_moment - V * V) / (L - 1)
    mc_error = 1.96 * np.sqrt(max(Var, 0.0))

    return V, mc_error

#1. Computation


### Computation of reference and coarse-grid CDF approximations

In this section we compute the reference cumulative distribution function together with several additional approximations obtained on coarser spatial grids.

First, the reference CDF is constructed using the fine-grid parameters $(m_{\mathrm{ref}}, d_{\mathrm{ref}})$.  
This reference solution is treated as the benchmark distribution in the subsequent experiments.

Next, several alternative CDF approximations are computed for different combinations of the FFT grid parameters $m$ and $d$.  
Recall that the total number of grid points is given by

$$
M = 2^m,
$$

while $d$ is the spatial step of the real-space grid.  
Thus, varying $m$ changes the grid resolution through the number of FFT nodes, whereas varying $d$ changes the spacing of the approximation points on the $x$-axis.

The code evaluates three families of grids:

$$
d = 5 \cdot 10^{-3}, \qquad m = 12, 13, 14,
$$

$$
d = 10^{-3}, \qquad m = 14, 15, 16,
$$

$$
d = 5 \cdot 10^{-4}, \qquad m = 15, 16, 17.
$$

These parameter choices allow us to study how the numerical approximation of the distribution changes as the Fourier grid becomes finer.  
In particular, they provide a systematic comparison between rough, intermediate, and relatively accurate discretizations.



In [ ]:
cdf_ref, x_ref = compute_cdf(m_ref, d_ref, sigma, p, r, intensity, lp, lm, T)

cdf_12_5e3, x_12_5e3 = compute_cdf(m12, d5e3, sigma, p, r, intensity, lp, lm, T)
cdf_13_5e3, x_13_5e3 = compute_cdf(m13, d5e3, sigma, p, r, intensity, lp, lm, T)
cdf_14_5e3, x_14_5e3 = compute_cdf(m14, d5e3, sigma, p, r, intensity, lp, lm, T)

cdf_14_1e3, x_14_1e3 = compute_cdf(m14, d1e3, sigma, p, r, intensity, lp, lm, T)
cdf_15_1e3, x_15_1e3 = compute_cdf(m15, d1e3, sigma, p, r, intensity, lp, lm, T)
cdf_16_1e3, x_16_1e3 = compute_cdf(m16, d1e3, sigma, p, r, intensity, lp, lm, T)

cdf_15_5e4, x_15_5e4 = compute_cdf(m15, d5e4, sigma, p, r, intensity, lp, lm, T)
cdf_16_5e4, x_16_5e4 = compute_cdf(m16, d5e4, sigma, p, r, intensity, lp, lm, T)
cdf_17_5e4, x_17_5e4 = compute_cdf(m17, d5e4, sigma, p, r, intensity, lp, lm, T)

plt.figure(figsize=(8, 6))
plt.plot(x_15_5e4, cdf_15_5e4, label='Approximation curve', color='red')
plt.xlim(-2, 2)

plt.title('Cumulative Distribution Function')
plt.xlabel('x')
plt.ylabel('CDF')
plt.legend()
plt.grid(True)



### Output of model and experiment parameters

This block prints the main parameters used throughout the numerical experiments.

In [ ]:
print("PARAMETERS USED")
print("-"*80)

print("Model (Kou):")
print(f"  intensity = {intensity}")
print(f"  p (P[positive jump]) = {p}")
print(f"  lp (Lambda_+) = {lp}")
print(f"  lm (Lambda_-) = {lm}")
print()

print("Market/Option:")
print(f"  sigma = {sigma}")
print(f"  r = {r}")
print(f"  T = {T}")
print(f"  K = {K}")
print(f"  z (damping) = {z}")
print()

print("Seeds / MC:")
print(f"  seed = {seed}")
print(f"  Direct MC: M = {M}, confidence = {confidence}, lambda_p = {lambda_p}, lambda_m = {lambda_m}")
print(f"  Hybrid MC: L = {L}")
print()

print("NN training:")
print(f"  N = {N}")
print(f"  epochs = {epochs}")
print()

print("Reference grid:")
print(f"  m_ref = {m_ref}, d_ref = {d_ref}")
print()

print("Training grids:")
print(f"  d5e3 = {d5e3}, m = {[m12, m13, m14]}")
print(f"  d1e3 = {d1e3}, m = {[m14, m15, m16]}")
print(f"  d5e4 = {d5e4}, m = {[m15, m16, m17]}")
print("-"*80)
print()

### Model tagging and experiment bookkeeping

This auxiliary block is used to organize and label neural-network training experiments.  
Since the approximation of the CDF is performed for multiple grid configurations, it is convenient to associate each trained model with the parameters used during its training.


In [ ]:

CURRENT_MODEL_TAG = ""
MODEL_BATCH = {}

def set_model_tag(m, d, batch_size):
    """
    Call this right before compute_approximation(...) for each model.
    Writes header to stdout (and thus into the report via Tee).
    """
    global CURRENT_MODEL_TAG
    CURRENT_MODEL_TAG = f"m={m}, d={d}, batch_size={batch_size}"
    MODEL_BATCH[(m, d)] = batch_size

    print()
    print("=" * 90)
    print("TRAIN MODEL:", CURRENT_MODEL_TAG)
    print("=" * 90)
    print()

### Training the neural approximations for all grid configurations

In this block we train the neural-network approximation of the reference CDF for all grid configurations considered in the numerical experiments.

Each run applies the procedure described earlier to a specific pair of discretization parameters $(m,d)$ and to a corresponding batch size.  
For every configuration, the training data consist of the grid points and the reference CDF values previously computed on that grid.  
The same network architecture is used throughout, with a fixed number of hidden units $N$ and a fixed number of training epochs.

Thus, for each experiment, we construct an approximation of the form

$$
\widehat{F}(x)
=
\sum_{j=1}^{N} p_j \, \sigma(\alpha_j x + \beta_j),
$$

where the parameters $\alpha_j$, $\beta_j$, and $p_j$ are learned separately for each grid resolution.

The purpose of this block is not to introduce a new method, but to repeat the training stage systematically over several discretizations.  
In this way, we obtain a family of neural approximations corresponding to progressively finer grids.  
The tested configurations are grouped into three regimes:

$$
d = 0.005, \qquad d = 0.001, \qquad d = 0.0005,
$$

with increasing values of $m$ inside each regime.  
Since the number of training points grows as

$$
M = 2^m,
$$

finer grids contain more samples and therefore require larger batch sizes during optimization.  
This is why the code increases the batch size as $m$ becomes larger.

For each configuration, the algorithm returns the trained parameters of the approximation, namely:
the first-layer weights $\alpha_j$, the biases $\beta_j$, the mixture probabilities $p_j$, the total training time, and the trained model itself.  
These outputs are stored separately for every pair $(m,d)$ so that they can later be used in the hybrid Monte Carlo pricing procedure.

Therefore, this block performs the full experimental training sweep and prepares all neural approximations needed for the subsequent comparison of pricing accuracy, statistical error, and computational cost across different grid resolutions.

In [ ]:
# 1) m=12, d=0.005, batch=32
set_model_tag(m12, d5e3, 32)
alphas12_5e3, betas12_5e3, probabilities12_5e3, train_time12_5e3, model12_5e3 = compute_approximation(
    x_train=x_12_5e3,
    y_train=np.array(cdf_12_5e3),
    epochs=epochs,
    batch_size=32,
    N=N,
    m=m12,
    d=d5e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

#2) m=13, d=0.005, batch=64
set_model_tag(m13, d5e3, 64)
alphas13_5e3, betas13_5e3, probabilities13_5e3, train_time13_5e3, model13_5e3 = compute_approximation(
    x_train=x_13_5e3,
    y_train=np.array(cdf_13_5e3),
    epochs=epochs,
    batch_size=64,
    N=N,
    m=m13,
    d=d5e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

#3) m=14, d=0.005, batch=128
set_model_tag(m14, d5e3, 128)
alphas14_5e3, betas14_5e3, probabilities14_5e3, train_time14_5e3, model14_5e3 = compute_approximation(
    x_train=x_14_5e3,
    y_train=np.array(cdf_14_5e3),
    epochs=epochs,
    batch_size=128,
    N=N,
    m=m14,
    d=d5e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

# 4) m=14, d=0.001, batch=128
set_model_tag(m14, d1e3, 128)
alphas14_1e3, betas14_1e3, probabilities14_1e3, train_time14_1e3, model14_1e3 = compute_approximation(
    x_train=x_14_1e3,
    y_train=np.array(cdf_14_1e3),
    epochs=epochs,
    batch_size=128,
    N=N,
    m=m14,
    d=d1e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)
# log_cdf_signed_error(x_14_1e3, model14_1e3, x_ref, cdf_ref)

# 5) m=15, d=0.001, batch=256
set_model_tag(m15, d1e3, 256)
alphas15_1e3, betas15_1e3, probabilities15_1e3, train_time15_1e3, model15_1e3 = compute_approximation(
    x_train=x_15_1e3,
    y_train=np.array(cdf_15_1e3),
    epochs=epochs,
    batch_size=256,
    N=N,
    m=m15,
    d=d1e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

# 6) m=16, d=0.001, batch=512
set_model_tag(m16, d1e3, 512)
alphas16_1e3, betas16_1e3, probabilities16_1e3, train_time16_1e3, model16_1e3 = compute_approximation(
    x_train=x_16_1e3,
    y_train=np.array(cdf_16_1e3),
    epochs=epochs,
    batch_size=512,
    N=N,
    m=m16,
    d=d1e3,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

# 7) m=15, d=0.0005, batch=256
set_model_tag(m15, d5e4, 256)
alphas15_5e4, betas15_5e4, probabilities15_5e4, train_time15_5e4, model15_5e4 = compute_approximation(
    x_train=x_15_5e4,
    y_train=np.array(cdf_15_5e4),
    epochs=epochs,
    batch_size=256,
    N=N,
    m=m15,
    d=d5e4,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

# 8) m=16, d=0.0005, batch=512
set_model_tag(m16, d5e4, 512)
alphas16_5e4, betas16_5e4, probabilities16_5e4, train_time16_5e4, model16_5e4 = compute_approximation(
    x_train=x_16_5e4,
    y_train=np.array(cdf_16_5e4),
    epochs=epochs,
    batch_size=512,
    N=N,
    m=m16,
    d=d5e4,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)

#9) m=17, d=0.0005, batch=1024
set_model_tag(m17, d5e4, 1024)
alphas17_5e4, betas17_5e4, probabilities17_5e4, train_time17_5e4, model17_5e4 = compute_approximation(
    x_train=x_17_5e4,
    y_train=np.array(cdf_17_5e4),
    epochs=epochs,
    batch_size=1024,
    N=N,
    m=m17,
    d=d5e4,
    m_ref=m_ref,
    d_ref=d_ref,
    sigma=sigma,
    p=p,
    r=r,
    intensity=intensity,
    lp=lp,
    lm=lm,
    T=T,
    seed=seed
)


### Evaluation of the neural CDF approximation error

In this block we evaluate the approximation error of the trained neural-network CDF for several grid configurations $(m,d)$.

For each pair of parameters, the function `compute_cdf_error` compares the neural approximation produced by the trained model with the reference cumulative distribution function previously obtained by the FFT-based inversion procedure.  
The comparison is performed against the same benchmark pair $(x_{\mathrm{ref}}, F_{\mathrm{ref}}(x))$, where $x_{\mathrm{ref}}$ is the reference spatial grid and $F_{\mathrm{ref}}$ is the corresponding reference CDF.

Thus, for every trained model, the quantity being analyzed is the deviation

$$
\widehat{F}(x) - F_{\mathrm{ref}}(x),
$$

where $\widehat{F}(x)$ is the neural approximation and $F_{\mathrm{ref}}(x)$ is the reference distribution.  
This allows us to assess how accurately the neural representation reproduces the benchmark CDF for different discretization parameters.


In [ ]:
_, _ = compute_cdf_error(
    x_train=x_12_5e3,
    model=model12_5e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m12,
    d=d5e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_13_5e3,
    model=model13_5e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m13,
    d=d5e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_14_5e3,
    model=model14_5e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m14,
    d=d5e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_14_1e3,
    model=model14_1e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m14,
    d=d1e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_15_1e3,
    model=model15_1e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m15,
    d=d1e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_16_1e3,
    model=model16_1e3,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m16,
    d=d1e3,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_15_5e4,
    model=model15_5e4,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m15,
    d=d5e4,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_16_5e4,
    model=model16_5e4,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m16,
    d=d5e4,
    N=N
)

_, _ = compute_cdf_error(
    x_train=x_17_5e4,
    model=model17_5e4,
    ref_x=x_ref,
    ref_cdf=cdf_ref,
    m=m17,
    d=d5e4,
    N=N
)





### Reference FFT prices on the test set of initial asset values

In this block we compute the benchmark option prices by applying the FFT-based pricing method to several values of the initial asset price $S_0$.

The set of test points is chosen as

$$
S_0 \in \{80, 85, 90, 95, 100, 105\}.
$$

For each value of $S_0$, the function `compute_price_fft` is called with the reference grid parameters $m_{\mathrm{ref}}$ and $d_{\mathrm{ref}}$.  
This produces a highly accurate numerical price of the European call option under the Kou model.

The resulting values are stored in the list `ref_prices_fft`.  
Each computed price is written in the form

$$
[V, V, V],
$$

that is, the same reference value is repeated three times.  
This format is used for consistency with the later calculations of errors.


In [ ]:
ref_prices_fft = []
S = [80, 85, 90, 95, 100, 105]
for S_0 in S:
    price = compute_price_fft(m_ref, d_ref, S_0, sigma, p, r, intensity, lp, lm, T, K, z)
    ref_prices_fft.append([price, price, price])

print(f"{'  Referenced FFT prices\n':>8}", end="\n")
for i, row in zip(S, ref_prices_fft):
  print(f"{i:>8}{row[0]:>12.6f}")

### FFT price comparison on coarse grids

In this block we compute FFT-based option prices for several coarser spatial grids and compare them across different values of the discretization parameters.

As before, the option prices are evaluated for the set of initial asset values

$$
S_0 \in \{80, 85, 90, 95, 100, 105\}.
$$

Instead of using only the reference grid, several parameter configurations are considered.  
The experiments are grouped by the spatial step $d$, and for each value of $d$ a collection of grid exponents $m$ is tested.  
Since the total number of grid points is

$$
M = 2^m,
$$

changing $m$ modifies the FFT resolution, while changing $d$ affects the spacing of the spatial grid.

The parameter sets examined in this block are

$$
d = 0.005, \qquad m \in \{12, 13, 14\},
$$

$$
d = 0.001, \qquad m \in \{14, 15, 16\},
$$

$$
d = 0.0005, \qquad m \in \{15, 16, 17\}.
$$

For each pair $(m,d)$ and for each value of $S_0$, the function `compute_price_fft` is applied to obtain the corresponding option price.  
The results are printed in tabular form, where each row corresponds to a value of $S_0$ and each column corresponds to a different grid parameter $m$ for the fixed step size $d$.


In [ ]:
S = [80, 85, 90, 95, 100, 105]

param_sets = [
    (d5e3, [12, 13, 14]),
    (d1e3, [14, 15, 16]),
    (d5e4, [15, 16, 17])
]

for d_val, m_list in param_sets:
    prices_fft = []

    for S_0 in S:
        current_prices_fft = []
        for m_ in m_list:
            price = compute_price_fft(m_, d_val, S_0, sigma, p, r, intensity, lp, lm, T, K, z)
            current_prices_fft.append(price)
        prices_fft.append(current_prices_fft)

    if d_val == d5e4:
        prices_fft_5e4 = [row[:] for row in prices_fft]
        m_list_fft_5e4 = m_list[:]

    print(f"\nFFT prices for d = {d_val}\n")
    print(f"{'S \\ m':>8}", end="")
    for j in m_list:
        print(f"{j:>12}", end="")
    print()

    for i, row in zip(S, prices_fft):
        print(f"{i:>8}", end="")
        for val in row:
            print(f"{val:>12.6f}", end="")
        print()

# --- Keeping some results for demonstration: m = 17, d = 0.0005
errors_FFT_abs_5e4 = [
    [fft - ref for ref, fft in zip(row_ref, row_fft)]
    for row_ref, row_fft in zip(ref_prices_fft, prices_fft_5e4)
]

errors_FFT_rel_5e4 = [
    [(fft - ref) / ref if ref != 0 else float("nan")
     for ref, fft in zip(row_ref, row_fft)]
    for row_ref, row_fft in zip(ref_prices_fft, prices_fft_5e4)
]

### Direct Monte Carlo price computation and error analysis

In this block we compute European call option prices by the direct Monte Carlo method for several values of the initial asset price $S_0$, and then compare the results with the reference FFT benchmarks.

The set of test values is

$$
S_0 \in \{80, 85, 90, 95, 100, 105\}.
$$

For each value of $S_0$, the function `compute_price_MC_direct` is called.  
This produces two quantities: the Monte Carlo price estimate and its statistical error.  
The latter reflects the sampling uncertainty of the estimator and is reported in parentheses next to the computed option value.

The execution time of each Monte Carlo experiment is also measured.  
This makes it possible to compare not only the pricing accuracy but also the computational cost of the direct Monte Carlo method across different initial asset prices.

For each $S_0$, the estimated price is stored in the form

$$
[V, V, V]
$$

This format is used for consistency with the later calculations of errors.

After the prices are computed, they are compared with the reference FFT prices.  


The absolute error is defined as

$$
\mathrm{AbsError}_{\mathrm{MC}}
=
V_{\mathrm{MC}} - V_{\mathrm{ref}},
$$

where $V_{\mathrm{ref}}$ is the benchmark price obtained from the reference FFT method.

The relative error is defined as

$$
\mathrm{RelError}_{\mathrm{MC}}
=
\frac{V_{\mathrm{MC}} - V_{\mathrm{ref}}}{V_{\mathrm{ref}}}
\qquad
$$

These quantities are printed for each value of $S_0$.  
Thus, this block provides a complete direct Monte Carlo benchmark, including computed prices, statistical confidence errors, runtime measurements, and both absolute and relative deviations from the reference FFT solution.

In [ ]:
import time

S = [80, 85, 90, 95, 100, 105]

prices_MC = []
times_MC = []
errors_MC = []

for S0 in S:
    start_time = time.perf_counter()
    price, error = compute_price_MC_direct(
    S0, K, r, T, sigma,
    intensity, p, lambda_p, lambda_m,
    M, confidence, seed
)
    end_time = time.perf_counter()
    times_MC.append(end_time - start_time)
    prices_MC.append([price, price, price])
    errors_MC.append(error)

print("Direct MC prices (with statistical errors in brackets)")
print(f"{'S0':>8}{'MC':>24}")
for s0, row, err in zip(S, prices_MC, errors_MC):
    cell = f"{row[0]:.6f} ({err:.6f})"
    print(f"{s0:>8}{cell:>24}")
print()

print(f"\n{'Execution time per experiment':>30}")
print(f"{'S0':>8}{'Time (sec)':>15}")
for s0, time in zip(S, times_MC):
  print(f"{s0:>8}{time:>15.6f}")


errors_MC_abs = [
    [mc - ref for ref, mc in zip(row_ref, row_mc)]
    for row_ref, row_mc in zip(ref_prices_fft, prices_MC)
]

errors_MC_rel = [
    [(mc - ref) / ref if ref != 0 else float("nan")
     for ref, mc in zip(row_ref, row_mc)]
    for row_ref, row_mc in zip(ref_prices_fft, prices_MC)
]

print("MC relative errors")
for i, row in zip(S, errors_MC_rel):
    print(f"{i:>8}{row[0]:>12.6f}")

print("MC absolute errors")
for i, row in zip(S, errors_MC_abs):
    print(f"{i:>8}{row[0]:>12.6f}")




### Hybrid Monte Carlo price computation and error analysis

In this block we compute European call option prices by the hybrid Monte Carlo method for several trained neural approximations and compare the results across different grid configurations.

The experiments are carried out for the set of initial asset values

$$
S_0 \in \{80, 85, 90, 95, 100, 105\}.
$$

The trained models are grouped according to the grid step used in the construction of the reference CDF and in the neural approximation.  
Three groups of discretization parameters are considered:

$$
d = 0.005 \quad \text{with} \quad m \in \{12,13,14\},
$$

$$
d = 0.001 \quad \text{with} \quad m \in \{14,15,16\},
$$

$$
d = 0.0005 \quad \text{with} \quad m \in \{15,16,17\}.
$$

For each pair $(m,d)$, the corresponding trained neural-network parameters
$\alpha_j$, $\beta_j$, and $p_j$ are taken from the previously constructed approximation

$$
\widehat{F}(x)
=
\sum_{j=1}^{N} p_j \, \sigma(\alpha_j x + \beta_j).
$$

Using these parameters, the function `compute_price_hybrid` is applied for every value of $S_0$.  
Thus, for each grid configuration and each initial asset price, the code computes:

1. the hybrid Monte Carlo price estimate,
2. the statistical Monte Carlo error,
3. the execution time of the pricing procedure.

The resulting prices are stored in matrix form, where rows correspond to different values of $S_0$ and columns correspond to different values of $m$ inside the fixed group of step sizes $d$.

After that, the hybrid Monte Carlo prices are compared with the benchmark FFT prices.  
Two error measures are formed.

The absolute error is defined as

$$
\mathrm{AbsError}_{\mathrm{HMC}}
=
V_{\mathrm{HMC}} - V_{\mathrm{ref}},
$$

and the relative error is

$$
\mathrm{RelError}_{\mathrm{HMC}}
=
\frac{V_{\mathrm{HMC}} - V_{\mathrm{ref}}}{V_{\mathrm{ref}}}
\qquad
$$

In addition, for each experiment the statistical Monte Carlo uncertainty is reported in brackets next to the corresponding HMC price.  
This allows one to distinguish between the deterministic approximation error caused by the neural representation and the sampling error caused by Monte Carlo simulation.

In [ ]:
import time

S = [80, 85, 90, 95, 100, 105]

def print_table(title, data):
    print(title)
    header = f"{'S \\ m':>8}"
    for j in m_list:
        header += f"{j:>12}"
    print(header)

models_hmc = [
    (12, '5e3', alphas12_5e3, betas12_5e3, probabilities12_5e3),
    (13, '5e3', alphas13_5e3, betas13_5e3, probabilities13_5e3),
    (14, '5e3', alphas14_5e3, betas14_5e3, probabilities14_5e3),

    (14, '1e3', alphas14_1e3, betas14_1e3, probabilities14_1e3),
    (15, '1e3', alphas15_1e3, betas15_1e3, probabilities15_1e3),
    (16, '1e3', alphas16_1e3, betas16_1e3, probabilities16_1e3),

    (15, '5e4', alphas15_5e4, betas15_5e4, probabilities15_5e4),
    (16, '5e4', alphas16_5e4, betas16_5e4, probabilities16_5e4),
    (17, '5e4', alphas17_5e4, betas17_5e4, probabilities17_5e4)
]


for d_label in ['5e3', '1e3', '5e4']:

    models_subset = [m for m in models_hmc if m[1] == d_label]

    m_list = [m_[0] for m_ in models_subset]
    alphas_list = [m_[2] for m_ in models_subset]
    betas_list = [m_[3] for m_ in models_subset]
    probs_list = [m_[4] for m_ in models_subset]

    prices_HMC = []
    errors_HMC_stat = []
    times_HMC = []

    for S_0 in S:

        current_prices_HMC = []
        current_errors_HMC_stat = []
        current_times_HMC = []

        for alphas_, betas_, probs_ in zip(alphas_list, betas_list, probs_list):

            start_time = time.perf_counter()
            price, error_HMC_stat = compute_price_hybrid(
                S_0, alphas_, betas_, probs_,
                K, T, L, r, seed
            )
            end_time = time.perf_counter()

            current_prices_HMC.append(price)
            current_errors_HMC_stat.append(error_HMC_stat)
            current_times_HMC.append(end_time - start_time)

        prices_HMC.append(current_prices_HMC)
        errors_HMC_stat.append(current_errors_HMC_stat)
        times_HMC.append(current_times_HMC)

    errors_HMC_abs = [
        [hmc - ref for ref, hmc in zip(row_ref, row_hmc)]
        for row_ref, row_hmc in zip(ref_prices_fft, prices_HMC)
    ]

    errors_HMC_rel = [
        [(hmc - ref) / ref if ref != 0 else float("nan")
         for ref, hmc in zip(row_ref, row_hmc)]
        for row_ref, row_hmc in zip(ref_prices_fft, prices_HMC)
    ]

    # --- Keeping some results for demonstration: m = 17, d = 0.0005
    if d_label == '5e4':
        prices_HMC_5e4 = [row[:] for row in prices_HMC]
        errors_HMC_stat_5e4 = [row[:] for row in errors_HMC_stat]
        errors_HMC_abs_5e4 = [row[:] for row in errors_HMC_abs]
        errors_HMC_rel_5e4 = [row[:] for row in errors_HMC_rel]
        times_HMC_5e4 = [row[:] for row in times_HMC]
        m_list_HMC_5e4 = m_list[:]



    print(f"\n\n=== HMC prices and errors for d = {d_label} ===\n")

    print("HMC prices (with statistical errors in brackets)")
    header = f"{'S \\ m':>8}"
    for j in m_list:
        header += f"{j:>24}"
    print(header)

    for s0, row_price, row_err in zip(S, prices_HMC, errors_HMC_stat):
        line = f"{s0:>8}"
        for price, err in zip(row_price, row_err):
            cell = f"{price:.6f} ({err:.6f})"
            line += f"{cell:>24}"
        print(line)
    print()

    print_table("HMC relative errors", errors_HMC_rel)
    print_table("HMC absolute errors", errors_HMC_abs)
    print_table("Execution time per experiment (sec)", times_HMC)


### Final comparison of pricing methods

This block prints the final comparison table of option prices obtained by different methods for $S_0 \in \{80,85,90,95,100,105\}$.  
The table reports the reference FFT price together with the coarse-grid FFT result ($m=17$, $d=0.0005$), the hybrid Monte Carlo estimate, and the direct Monte Carlo estimate, with statistical errors shown in parentheses for the Monte Carlo methods.

In [ ]:
print()
print("=" * 140)
print("FINAL COMPARISON TABLE: Reference | FFT (m=17, d=0.0005) | HMC (m=17, d=0.0005) | MC")
print("=" * 140)

header = (
    f"{'S0':>8}"
    f"{'Reference FFT':>20}"
    f"{'FFT m=17,d=0.0005':>22}"
    f"{'HMC m=17,d=0.0005':>30}"
    f"{'Direct MC':>24}"
)
print(header)

for i, s0 in enumerate(S):
    ref_val = float(ref_prices_fft[i][0])
    fft17_val = float(prices_fft_5e4[i][2])
    hmc17_val = float(prices_HMC_5e4[i][0])
    hmc17_err = float(errors_HMC_stat_5e4[i][0])
    mc_val = float(prices_MC[i][0])
    mc_err = float(errors_MC[i])

    line = (
        f"{s0:>8}"
        f"{ref_val:>20.6f}"
        f"{fft17_val:>22.6f}"
        f"{f'{hmc17_val:.6f} ({hmc17_err:.6f})':>30}"
        f"{f'{mc_val:.6f} ({mc_err:.6f})':>24}"
    )
    print(line)

print("=" * 140)
print()

### Error comparison of pricing methods

This block prints a table comparing the absolute and relative errors of the FFT, hybrid Monte Carlo, and direct Monte Carlo pricing methods relative to the reference FFT benchmark.  
For each value of $S_0$, the errors are computed as differences between the corresponding method price and the reference price.

In [ ]:
print("=" * 190)
print("ERROR COMPARISON TABLE: FFT | HMC | MC")
print("=" * 190)

header = (
    f"{'S0':>8}"
    f"{'FFT rel error':>20}"
    f"{'FFT abs error':>20}"
    f"{'HMC abs error':>20}"
    f"{'HMC rel error':>20}"
    f"{'MC abs error':>20}"
    f"{'MC rel error':>20}"
)
print(header)

for i, s0 in enumerate(S):
    fft_rel = float(errors_FFT_rel_5e4[i][2])
    fft_abs = float(errors_FFT_abs_5e4[i][2])
    hmc_abs = float(errors_HMC_abs_5e4[i][0])
    hmc_rel = float(errors_HMC_rel_5e4[i][0])
    mc_abs = float(errors_MC_abs[i][0])
    mc_rel = float(errors_MC_rel[i][0])

    line = (
        f"{s0:>8}"
        f"{fft_rel:>20.6f}"
        f"{fft_abs:>20.6f}"
        f"{hmc_abs:>20.6f}"
        f"{hmc_rel:>20.6f}"
        f"{mc_abs:>20.6f}"
        f"{mc_rel:>20.6f}"
    )
    print(line)

print("=" * 190)
print()

In [ ]:
# --- Closing report-file and downloading
sys.stdout = _old_stdout
_report_f.close()
files.download(REPORT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>